# CASMI 2026: EXP029 offline inference

This notebook performs inference only. It does not access the internet, retrain models, or download data. Attach the competition data, the updated EXP029 source bundle, and the frozen EXP027 asset bundle before running all cells. The final file is `/kaggle/working/submission.csv`.

In [ ]:
import os
import sys
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/casmi26-exp029')
OUTPUT = Path('/kaggle/working/submission.csv')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('Attached datasets:', sorted(path.name for path in INPUT_ROOT.iterdir()))

In [ ]:
import importlib.metadata
import subprocess

def find_attached_file(filename, sibling=None):
    matches = sorted(INPUT_ROOT.rglob(filename))
    if sibling is not None:
        matches = [path for path in matches if (path.parent / sibling).exists()]
    if len(matches) != 1:
        attached = sorted(path.name for path in INPUT_ROOT.iterdir())
        raise FileNotFoundError(
            f'Expected exactly one {filename!r} under {INPUT_ROOT}, found {matches}. '
            f'Attached dataset roots: {attached}'
        )
    return matches[0]

runner = find_attached_file('run_offline_inference.py')
test_path = find_attached_file('test.parquet', sibling='sample_submission.csv')
spectral_index = find_attached_file('pipeline_compact.joblib')
PROJECT_ROOT = Path(os.environ.get('CASMI_PROJECT_ROOT', runner.parent.parent))
COMPETITION_ROOT = Path(os.environ.get('CASMI_COMPETITION_ROOT', test_path.parent))
ASSET_ROOT = Path(os.environ.get('CASMI_ASSET_ROOT', spectral_index.parent))
paths = {
    'runner': runner,
    'test': test_path,
    'sample_submission': test_path.parent / 'sample_submission.csv',
    'candidate_db': ASSET_ROOT / 'expanded_candidates.parquet',
    'fingerprint_model': ASSET_ROOT / 'fingerprint_model.pt',
    'fingerprint_index': ASSET_ROOT / 'expanded_morgan_2048.joblib',
    'spectral_index': spectral_index,
    'analog_index': ASSET_ROOT / 'raw_entropy_representatives.joblib',
    'descriptor_cache': ASSET_ROOT / 'expanded_candidate_descriptors.parquet',
    'ranker_dir': ASSET_ROOT / 'ranker',
}
missing = {name: str(path) for name, path in paths.items() if not path.exists()}
assert not missing, f'Missing attached offline assets: {missing}'
print({name: str(path) for name, path in paths.items()})
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['PYTHONPATH'] = os.pathsep.join(value for value in (str(PROJECT_ROOT / 'src'), os.environ.get('PYTHONPATH', '')) if value)
try:
    installed_rdkit = importlib.metadata.version('rdkit')
except importlib.metadata.PackageNotFoundError:
    installed_rdkit = None
if installed_rdkit != '2026.3.3':
    wheel_dir = PROJECT_ROOT / 'wheels'
    wheels = sorted(wheel_dir.glob('rdkit-2026.3.3-*.whl'))
    assert wheels, f'Missing offline RDKit wheels under {wheel_dir}'
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
        '--find-links', str(wheel_dir), 'rdkit==2026.3.3',
    ], check=True)
import lightgbm, numba, pyarrow, rdkit, torch  # noqa: F401
print('Offline asset preflight passed; RDKit', rdkit.__version__)

In [ ]:
import subprocess

command = [
    sys.executable, str(paths['runner']),
    '--project-root', str(PROJECT_ROOT),
    '--test', str(paths['test']),
    '--sample-submission', str(paths['sample_submission']),
    '--candidate-db', str(paths['candidate_db']),
    '--fingerprint-model', str(paths['fingerprint_model']),
    '--fingerprint-index', str(paths['fingerprint_index']),
    '--spectral-index', str(paths['spectral_index']),
    '--analog-index', str(paths['analog_index']),
    '--descriptor-cache', str(paths['descriptor_cache']),
    '--ranker-dir', str(paths['ranker_dir']),
    '--work-dir', str(WORK_ROOT),
    '--output', str(OUTPUT),
]
subprocess.run(command, check=True)

In [ ]:
import pandas as pd

submission = pd.read_csv(OUTPUT)
sample = pd.read_csv(paths['sample_submission'])
counts = submission['smiles'].str.split(';').str.len()
assert list(submission.columns) == ['molecule_id', 'smiles']
assert set(submission['molecule_id'].astype(str)) == set(sample['molecule_id'].astype(str))
assert counts.between(1, 25).all()
assert not submission.isna().any().any()
print({'rows': len(submission), 'min_candidates': int(counts.min()), 'max_candidates': int(counts.max()), 'output': str(OUTPUT)})
submission.head()